# FLAN-T5 generation and evaluation

This notebook runs the existing `ai_models/flan_t5.py` implementation without changing its prompt, model, or generation settings. It uses `full_body` from `output/scraped_news.json` and stores all generated artifacts in `notebook/flan_t5_results/`.

ROUGE and BERTScore require a human-written reference summary. On the first run, `reference_summary` is intentionally empty. Fill that column in `flan_summaries.csv` from the real article before running the two evaluation sections.

## 1. Optional evaluation dependency installation

Run the next command only if `rouge_score` or `bert_score` is missing from the active notebook kernel. Restart the kernel after installation if imports still fail.

In [ ]:
# %pip install bert-score==0.3.13 rouge-score==0.1.2

## 2. Locate the project and configure output paths

In [ ]:
from __future__ import annotations

import hashlib
import inspect
import json
import re
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'ai_models' / 'flan_t5.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root containing ai_models/flan_t5.py')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_PATH = PROJECT_ROOT / 'output' / 'scraped_news.json'
RESULTS_DIR = PROJECT_ROOT / 'notebook' / 'flan_t5_results'
SUMMARIES_PATH = RESULTS_DIR / 'flan_summaries.csv'
ROUGE_PATH = RESULTS_DIR / 'rouge_scores.csv'
BERTSCORE_PATH = RESULTS_DIR / 'bertscore_scores.csv'
EVALUATION_SUMMARY_PATH = RESULTS_DIR / 'evaluation_summary.csv'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Dataset:', DATASET_PATH)
print('Results directory:', RESULTS_DIR)

## 3. Load and validate real scraped articles

Only successful records with a non-empty string `full_body` are used. No synthetic article is created, and this notebook does not rescrape or silently clean article text.

In [ ]:
with DATASET_PATH.open('r', encoding='utf-8') as handle:
    scraped_records = json.load(handle)

if not isinstance(scraped_records, list):
    raise TypeError('scraped_news.json must contain a top-level JSON list')

valid_records = [
    record
    for record in scraped_records
    if isinstance(record, dict)
    and record.get('scrape_status') == 'success'
    and isinstance(record.get('full_body'), str)
    and record['full_body'].strip()
]

if not valid_records:
    raise ValueError('No successful records with a non-empty full_body were found')

article_ids = [str(record.get('guid') or record.get('url') or index) for index, record in enumerate(valid_records)]
if len(article_ids) != len(set(article_ids)):
    raise ValueError('Article identifiers are not unique')

body_lengths = pd.Series([len(record['full_body']) for record in valid_records])
print(f'Valid articles: {len(valid_records)} / {len(scraped_records)}')
print(f'full_body characters — min: {body_lengths.min()}, median: {body_lengths.median():.0f}, max: {body_lengths.max()}')
display(pd.DataFrame(valid_records)[['guid', 'source', 'title', 'scrape_status']].head())

## 4. Import the exact FLAN implementation

The notebook imports the repository service instead of copying model logic. This guarantees that the evaluated implementation is the same one used by the pipeline. The second line displays that source for inspection.

In [ ]:
from ai_models.flan_t5 import (
    DEFAULT_MAX_INPUT_TOKENS,
    DEFAULT_MODEL_NAME,
    SUMMARY_PROMPT,
    FlanT5Service,
)

print(inspect.getsource(FlanT5Service))

## 5. Load FLAN-T5 once

The service uses `google/flan-t5-base`, a 512-token maximum input, and the current deterministic generation settings from `ai_models/flan_t5.py`: four beams, at most 64 new tokens, early stopping, and no repeated 3-grams.

In [ ]:
flan_service = FlanT5Service()
flan_service.load()

print('Checkpoint:', flan_service.model_name)
print('Maximum input tokens:', flan_service.max_input_tokens)
print('Resolved device:', flan_service.device)
print('Generation: deterministic beam search, 4 beams, max 64 new tokens')

## 6. Generate summaries from `full_body`

Leave `ARTICLE_LIMIT = None` to evaluate every valid scraped article. Set a small integer only for a quick test. Existing human reference summaries are preserved when this cell is rerun.

In [ ]:
ARTICLE_LIMIT: int | None = None
selected_records = valid_records if ARTICLE_LIMIT is None else valid_records[:ARTICLE_LIMIT]

existing_references: dict[str, str] = {}
if SUMMARIES_PATH.exists():
    existing_df = pd.read_csv(SUMMARIES_PATH, dtype=str).fillna('')
    if {'article_id', 'reference_summary'}.issubset(existing_df.columns):
        existing_references = dict(zip(existing_df['article_id'], existing_df['reference_summary']))

summary_rows = []
for index, article in enumerate(tqdm(selected_records, desc='Generating FLAN summaries')):
    article_id = str(article.get('guid') or article.get('url') or index)
    full_body = article['full_body'].strip()
    try:
        result = flan_service.summarize_with_metrics(full_body)
        generated_summary = result.summary
        error = ''
        inference_seconds = result.inference_seconds
        input_token_count = result.input_token_count
        original_input_token_count = result.original_input_token_count
        output_token_count = result.output_token_count
        input_was_truncated = result.input_was_truncated
    except (TypeError, ValueError, RuntimeError) as exc:
        generated_summary = ''
        error = f'{type(exc).__name__}: {exc}'
        inference_seconds = None
        input_token_count = None
        original_input_token_count = None
        output_token_count = None
        input_was_truncated = None

    summary_rows.append({
        'article_id': article_id,
        'source': article.get('source', ''),
        'title': article.get('title', ''),
        'url': article.get('url', ''),
        'full_body': full_body,
        'generated_summary': generated_summary,
        'summary_sha256': hashlib.sha256(generated_summary.encode('utf-8')).hexdigest() if generated_summary else '',
        'reference_summary': existing_references.get(article_id, ''),
        'inference_seconds': inference_seconds,
        'input_token_count': input_token_count,
        'original_input_token_count': original_input_token_count,
        'output_token_count': output_token_count,
        'input_was_truncated': input_was_truncated,
        'generation_error': error,
    })

summaries_df = pd.DataFrame(summary_rows)
summaries_df.to_csv(SUMMARIES_PATH, index=False)
print(f'Saved {len(summaries_df)} rows to {SUMMARIES_PATH}')
display(summaries_df[['article_id', 'title', 'generated_summary', 'input_was_truncated', 'inference_seconds']])

## 7. Inspect generation measurements

These measurements do not require a reference summary. Truncation is especially important because FLAN receives only the first 512 prompt-and-article tokens.

In [ ]:
summaries_df = pd.read_csv(SUMMARIES_PATH).fillna('')
successful_df = summaries_df[summaries_df['generated_summary'].astype(str).str.strip().ne('')].copy()

generation_overview = pd.DataFrame({
    'metric': [
        'article_count',
        'successful_summary_count',
        'generation_error_count',
        'truncation_rate',
        'mean_inference_seconds',
        'mean_output_tokens',
    ],
    'value': [
        len(summaries_df),
        len(successful_df),
        int(summaries_df['generation_error'].astype(str).str.strip().ne('').sum()),
        pd.to_numeric(successful_df['input_was_truncated'], errors='coerce').mean(),
        pd.to_numeric(successful_df['inference_seconds'], errors='coerce').mean(),
        pd.to_numeric(successful_df['output_token_count'], errors='coerce').mean(),
    ],
})
display(generation_overview)

## 8. Add human reference summaries before scoring

Open `notebook/flan_t5_results/flan_summaries.csv` in a spreadsheet and fill `reference_summary` for each article. Each reference must be an objective, factual 1–2 sentence summary written from `full_body`. Do not copy the FLAN result. Save the CSV, then run the next cell to verify readiness.

In [ ]:
summaries_df = pd.read_csv(SUMMARIES_PATH, dtype=str).fillna('')
evaluation_ready_df = summaries_df[
    summaries_df['generated_summary'].str.strip().ne('')
    & summaries_df['reference_summary'].str.strip().ne('')
].copy()

print(f'References completed: {len(evaluation_ready_df)} / {len(summaries_df)}')
if evaluation_ready_df.empty:
    print('ROUGE and BERTScore are not ready. Fill reference_summary in flan_summaries.csv first.')
else:
    display(evaluation_ready_df[['article_id', 'title', 'generated_summary', 'reference_summary']].head())

## 9. ROUGE evaluation — separate cell and output

ROUGE-1 measures word overlap, ROUGE-2 measures two-word phrase overlap, and ROUGE-L measures longest-sequence overlap. Higher F1 is better, but ROUGE alone does not prove factual correctness.

In [ ]:
from rouge_score import rouge_scorer

if evaluation_ready_df.empty:
    print('Skipped: no completed human reference summaries.')
else:
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_rows = []
    for _, row in evaluation_ready_df.iterrows():
        scores = rouge.score(row['reference_summary'], row['generated_summary'])
        rouge_rows.append({
            'article_id': row['article_id'],
            'title': row['title'],
            'generated_summary': row['generated_summary'],
            'reference_summary': row['reference_summary'],
            'summary_sha256': row['summary_sha256'],
            'rouge1_precision': scores['rouge1'].precision,
            'rouge1_recall': scores['rouge1'].recall,
            'rouge1_f1': scores['rouge1'].fmeasure,
            'rouge2_precision': scores['rouge2'].precision,
            'rouge2_recall': scores['rouge2'].recall,
            'rouge2_f1': scores['rouge2'].fmeasure,
            'rougeL_precision': scores['rougeL'].precision,
            'rougeL_recall': scores['rougeL'].recall,
            'rougeL_f1': scores['rougeL'].fmeasure,
        })

    rouge_df = pd.DataFrame(rouge_rows)
    rouge_df.to_csv(ROUGE_PATH, index=False)
    print('Saved:', ROUGE_PATH)
    display(rouge_df)
    display(rouge_df[['rouge1_f1', 'rouge2_f1', 'rougeL_f1']].mean().to_frame('mean_score'))

## 10. BERTScore evaluation — separate cell and output

BERTScore compares contextual meaning rather than exact wording. The first run may download the standard English encoder and take several minutes. Higher precision, recall, and F1 are better, but factual review is still required.

In [ ]:
from bert_score import BERTScorer

if evaluation_ready_df.empty:
    print('Skipped: no completed human reference summaries.')
else:
    if torch.cuda.is_available():
        bertscore_device = 'cuda'
    elif torch.backends.mps.is_available():
        bertscore_device = 'mps'
    else:
        bertscore_device = 'cpu'

    bert_scorer = BERTScorer(lang='en', rescale_with_baseline=True, device=bertscore_device)
    precision, recall, f1 = bert_scorer.score(
        evaluation_ready_df['generated_summary'].tolist(),
        evaluation_ready_df['reference_summary'].tolist(),
        batch_size=16,
    )

    bertscore_df = evaluation_ready_df[
        ['article_id', 'title', 'generated_summary', 'reference_summary', 'summary_sha256']
    ].copy()
    bertscore_df['bertscore_precision'] = precision.cpu().tolist()
    bertscore_df['bertscore_recall'] = recall.cpu().tolist()
    bertscore_df['bertscore_f1'] = f1.cpu().tolist()
    bertscore_df['bertscore_model_hash'] = str(bert_scorer.hash)
    bertscore_df.to_csv(BERTSCORE_PATH, index=False)

    print('Device:', bertscore_device)
    print('Model hash:', bert_scorer.hash)
    print('Saved:', BERTSCORE_PATH)
    display(bertscore_df)
    display(bertscore_df[['bertscore_precision', 'bertscore_recall', 'bertscore_f1']].mean().to_frame('mean_score'))

## 11. Combine aggregate scores for analysis

This cell reads the two separate score files and creates `evaluation_summary.csv`. It checks summary fingerprints to prevent accidentally combining old scores with newly generated summaries.

In [ ]:
current_df = pd.read_csv(SUMMARIES_PATH, dtype=str).fillna('')
current_hashes = set(current_df.loc[current_df['summary_sha256'].ne(''), 'summary_sha256'])
aggregate_rows = []

if ROUGE_PATH.exists():
    saved_rouge_df = pd.read_csv(ROUGE_PATH, dtype={'summary_sha256': str})
    if not set(saved_rouge_df['summary_sha256']).issubset(current_hashes):
        raise ValueError('rouge_scores.csv is stale. Rerun the ROUGE cell.')
    for metric in ['rouge1_f1', 'rouge2_f1', 'rougeL_f1']:
        aggregate_rows.append({
            'evaluation': 'ROUGE',
            'metric': metric,
            'mean_score': pd.to_numeric(saved_rouge_df[metric], errors='coerce').mean(),
            'evaluated_articles': len(saved_rouge_df),
        })

if BERTSCORE_PATH.exists():
    saved_bertscore_df = pd.read_csv(BERTSCORE_PATH, dtype={'summary_sha256': str})
    if not set(saved_bertscore_df['summary_sha256']).issubset(current_hashes):
        raise ValueError('bertscore_scores.csv is stale. Rerun the BERTScore cell.')
    for metric in ['bertscore_precision', 'bertscore_recall', 'bertscore_f1']:
        aggregate_rows.append({
            'evaluation': 'BERTScore',
            'metric': metric,
            'mean_score': pd.to_numeric(saved_bertscore_df[metric], errors='coerce').mean(),
            'evaluated_articles': len(saved_bertscore_df),
        })

if not aggregate_rows:
    print('No ROUGE or BERTScore outputs exist yet. Complete references and run both evaluation cells.')
else:
    evaluation_summary_df = pd.DataFrame(aggregate_rows)
    evaluation_summary_df.to_csv(EVALUATION_SUMMARY_PATH, index=False)
    print('Saved:', EVALUATION_SUMMARY_PATH)
    display(evaluation_summary_df)

## Interpretation reminder

Use ROUGE to analyze wording/content overlap and BERTScore to analyze semantic similarity. Neither metric establishes factuality. Read each summary beside `full_body` and prioritize factual correctness, entity/number preservation, beginner clarity, objectivity, and the absence of investment advice or price prediction.